# Project 9 — Phase 2: M4/M5 public-source reproduction recipe

**SOURCE REVIEW — NOT A COMPLETED END-TO-END REPRODUCTION.**

Derived from the supplied `Project9_Phase2_CORRECTED_v3.ipynb` (107 cells, 82 code cells). The original notebook contains private extraction parameters, outdated status text, saved outputs and an **enabled final-Test cell**. Do **not** publish or Run All on the original.

This clean package provides frozen split metadata, safe label-building source for **TRAIN/VAL only**, guarded training recipes, a VAL-only evaluation example, and historic training CSV evidence. It **does not contain model weights, images, source annotations, credentials, URLs with personal usernames, original absolute Drive paths, or any code that performs final Test evaluation**.

**Known provenance:** original frozen split manifest SHA256 `ef6b35d2ba7f6708319b824e29707b4d2001750974ac3c4da03c72c9340fdb50`; sanitized metadata SHA256 `f88dee92e597a865e2fbc848fbaef98fc5e2d7c8c1344c7899b414817c5d1872`. They differ intentionally because runtime paths were removed. The public copy is not a byte-identical original manifest.

Current user results should be read from `evidence/`, *not* inferred by executing any notebook cell. Training replay may yield different results depending on hardware/package versions and datasets; no replay has been performed by the assistant.


## 1 — Environment and authorization

Obtain the original **ZeroWaste-f** data from its official source under its applicable license/terms, and extract locally using **your own authorized method**. This notebook does not include download URLs or archive passwords and never writes Test predictions. To use on Colab, unzip this source folder there, select its directory as the working directory, and install requirements. Colab Drive mounting is optional and not performed by these cells.

Run install separately if needed: `%pip install -r requirements.txt`.


In [ ]:
from pathlib import Path
import json, os, hashlib
import pandas as pd
from phase2_data_prep import manifest_check, build_detection, build_segmentation

BUNDLE = Path.cwd()
PUBLIC_MANIFEST = BUNDLE / "split_manifest_public.csv"
SOURCE = Path(os.environ.get("ZEROWASTE_SOURCE_ROOT", "/content/ZeroWaste/splits_final_deblurred"))
WORK = Path(os.environ.get("PROJECT9_WORK_DIR", "/content/Project9_Phase2"))
DET_DATA = WORK / "zerowaste_yolo_grouped_v2"
SEG_DATA = WORK / "zerowaste_seg_grouped_v2"
RUNS = WORK / "runs_reproduction"
assert PUBLIC_MANIFEST.is_file(), "Open this notebook from the unzipped public source folder"
manifest = manifest_check(PUBLIC_MANIFEST)
print("Local source root (not in repository):", SOURCE)
print("No training or Test evaluation has run.")

## 2 — Read the historic evidence (no model execution)

These are copied measurements of the original runs; **VAL training metrics are not the frozen final Test metrics**. M4 ran for 34 epochs. M5 ran for 36 epochs including the resumed portion. Saved `M5` `args.yaml` refers to `last.pt` (resume), while the original initial-training cell used `yolo11n-seg.pt`. Public configs intentionally omit original absolute paths.

In [ ]:
for name, metric in [("M4", "metrics/mAP50-95(B)"), ("M5", "metrics/mAP50-95(M)")]:
    file = BUNDLE / "evidence" / f"{name}_training_results.csv"
    results = pd.read_csv(file)
    results.columns = results.columns.str.strip()
    idx = results[metric].idxmax()
    print(name, "historical epochs:", len(results),
          "best recorded epoch:", int(results.loc[idx, "epoch"]),
          "best VAL metric:", round(float(results.loc[idx, metric]), 6))
print("Frozen Test values belong to the separately maintained report, not this reproduction.")

## 3 — Build grouped labels (TRAIN/VAL only; OFF by default)

This uses **exact frozen split rows** from `split_manifest_public.csv`. It does not recompute the split, download data, copy Test images, or create a `test:` key in YOLO's `data.yaml`. On first build it checks labels against the frozen counts and refuses to replace an existing directory with unexpected provenance. Source ZeroWaste-f COCO annotations and PNG images must exist outside GitHub. Input preparation reproduces the original label-conversion logic; exact end-to-end equivalence is not yet tested.

In [ ]:
BUILD_DATASETS = False  # Set to True only with an authorized local source dataset.
if BUILD_DATASETS:
    print("M4 TRAIN/VAL:", build_detection(manifest, SOURCE, DET_DATA, PUBLIC_MANIFEST))
    print("M5 TRAIN/VAL:", build_segmentation(manifest, SOURCE, SEG_DATA, PUBLIC_MANIFEST))
else:
    print("Skipped dataset building: no data copied and no Test touched.")

## 4 — New reproduction training (OFF by default)

**Do not rerun the historical M4 or M5 run.** This is a recipe for a separate experiment, with fresh names in `runs_reproduction`. It uses GPU 0, seed 42, YOLO11n/YOLO11n-seg, AdamW, 50-epoch maximum, and the recorded training configuration. Logs from such a run **are not** the original trained outputs. Original M5 was later resumed from checkpoint, and this script does not reconstruct its interruption/restart chronology.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    import torch
    from ultralytics import YOLO
    if not torch.cuda.is_available():
        raise RuntimeError("Training needs a configured GPU; nothing started")
    RUNS.mkdir(parents=True, exist_ok=True)
    common = dict(epochs=50, patience=15, imgsz=640, batch=8, device=0,
                  workers=2, optimizer="AdamW", lr0=0.001, seed=42,
                  deterministic=True, amp=True, save_period=5, close_mosaic=10,
                  cache=False, plots=True, project=str(RUNS), exist_ok=False)
    for label, model_name, dataset in [
        ("M4_detection_REPRO_NEW", "yolo11n.pt", DET_DATA),
        ("M5_segmentation_REPRO_NEW", "yolo11n-seg.pt", SEG_DATA),
    ]:
        if (RUNS/label).exists():
            raise FileExistsError(f"Will not overwrite an existing run: {RUNS/label}")
        if not (dataset/"data.yaml").is_file():
            raise FileNotFoundError(f"Build TRAIN/VAL data first: {dataset}")
        YOLO(model_name).train(data=str(dataset/"data.yaml"), name=label, **common)
else:
    print("Training disabled; original M4/M5 checkpoints are untouched.")

## 5 — Optional **Validation-only** evaluation (OFF by default)

Supply an existing checkpoint via environment variables `M4_CHECKPOINT` / `M5_CHECKPOINT`. The code validates only the `val` split and writes to a fresh reproduction folder. Frozen Test evaluation is absent from this notebook on purpose. Do not infer final Test performance from training curves.

In [ ]:
RUN_VALIDATION = False
if RUN_VALIDATION:
    from ultralytics import YOLO
    for label, env, dataset in [
        ("M4", "M4_CHECKPOINT", DET_DATA),
        ("M5", "M5_CHECKPOINT", SEG_DATA),
    ]:
        checkpoint = Path(os.environ[env])
        if not checkpoint.is_file():
            raise FileNotFoundError(checkpoint)
        if not (dataset/"data.yaml").is_file():
            raise FileNotFoundError(dataset/"data.yaml")
        output = WORK/"evaluation_reproduction_VAL"/label
        if output.exists():
            raise FileExistsError(f"Will not overwrite existing validation output: {output}")
        r = YOLO(str(checkpoint)).val(data=str(dataset/"data.yaml"),
                    split="val", imgsz=640, conf=0.001, iou=0.5,
                    project=str(output.parent), name=label, exist_ok=False)
        print(label, "VAL mAP50/95:",
              (float(r.box.map50), float(r.box.map)) if label == "M4"
              else (float(r.seg.map50), float(r.seg.map)))
else:
    print("Validation disabled; final frozen Test is never evaluated here.")

## 6 — M8 and extended analyses

The source-only M8 service, release-manifest reader, TACO domain shift, VAL count audit, CLIP and retrieval experiments have separate saved evidence and are **not included** in this M4/M5 training-source subset. Refer to the reviewed Phase 2 GitHub source and the 16-page final report for their scope. No M8 inference, open-world experiment, or Test result is claimed by this notebook.

**Publication gate:** inspect the `GIT_UPLOAD_CHECKLIST_AR.md` and team approval before pushing. The original executed 107-cell notebook must remain private.